# Data Cleaning
UFC fight stats dataset — cleaning before modeling.

## 1. Load Data

In [4]:
%pip install pandas numpy matplotlib seaborn scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [7]:
import pandas as pd
import numpy as np

fights = pd.read_csv('../data/raw/ufc_event_fight_stats.csv')
fighters = pd.read_csv('../data/raw/ufc_fighters.csv')
events = pd.read_csv('../data/raw/ufc_events.csv')

print(fights.shape, fighters.shape, events.shape)

(7840, 58) (4262, 19) (711, 4)


## 2. Initial Overview
Get a feel for the data — column types, missing values, basic stats.

In [8]:
# Check dtypes and non-null counts
fights.info()

<class 'pandas.DataFrame'>
RangeIndex: 7840 entries, 0 to 7839
Data columns (total 58 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   f1_id                7840 non-null   int64  
 1   f2_id                7840 non-null   int64  
 2   f1_name              7840 non-null   str    
 3   f2_name              7840 non-null   str    
 4   weight_class         7840 non-null   str    
 5   f1_age_during        7790 non-null   float64
 6   f2_age_during        7710 non-null   float64
 7   f1_height_cm         7834 non-null   float64
 8   f2_height_cm         7829 non-null   float64
 9   f1_knockdowns        7840 non-null   int64  
 10  f2_knockdowns        7840 non-null   int64  
 11  f1_sig_strike_atts   7840 non-null   int64  
 12  f2_sig_strike_atts   7840 non-null   int64  
 13  f1_sig_strikes       7840 non-null   int64  
 14  f2_sig_strikes       7840 non-null   int64  
 15  f1_tot_strike_atts   7840 non-null   int64  
 16 

In [9]:
# Count missing values per column
fights.isnull().sum()

f1_id                    0
f2_id                    0
f1_name                  0
f2_name                  0
weight_class             0
f1_age_during           50
f2_age_during          130
f1_height_cm             6
f2_height_cm            11
f1_knockdowns            0
f2_knockdowns            0
f1_sig_strike_atts       0
f2_sig_strike_atts       0
f1_sig_strikes           0
f2_sig_strikes           0
f1_tot_strike_atts       0
f2_tot_strike_atts       0
f1_tot_strikes           0
f2_tot_strikes           0
f1_takedown_atts         0
f2_takedown_atts         0
f1_takedowns             0
f2_takedowns             0
f1_submissions           0
f2_submissions           0
f1_reversals             0
f2_reversals             0
f1_ctrl_time           120
f2_ctrl_time           120
f1_head_strike_atts      0
f2_head_strike_atts      0
f1_head_strikes          0
f2_head_strikes          0
f1_body_strike_atts      0
f2_body_strike_atts      0
f1_body_strikes          0
f2_body_strikes          0
f

In [17]:
# Quick look at a few rows
fights.head()

,f1_id,f2_id,f1_name,f2_name,weight_class,f1_age_during,f2_age_during,f1_height_cm,f2_height_cm,f1_knockdowns,...,f2_clinchs,f1_ground_atts,f2_ground_atts,f1_grounds,f2_grounds,f1_total_fight_time,f2_total_fight_time,result,fights_url,event_url
0,3363,2768,Cory Sandhagen,Umar Nurmagomedov,Bantamweight Bout,32.0,28.0,180.34,172.72,0,...,2,0,0,0,0,1500,1500,2768,http://www.ufcstats.com/fight-details/33655fd0...,http://www.ufcstats.com/event-details/fe435e44...
1,2275,2800,Shara Magomedov,Michal Oleksiejczuk,Middleweight Bout,30.0,29.0,187.96,182.88,0,...,4,0,2,0,1,900,900,2275,http://www.ufcstats.com/fight-details/b38450d9...,http://www.ufcstats.com/event-details/fe435e44...
2,1576,299,Jai Herbert,Rolando Bedoya,Lightweight Bout,36.0,27.0,185.42,180.34,1,...,3,1,0,0,0,900,900,1576,http://www.ufcstats.com/fight-details/d038ff5a...,http://www.ufcstats.com/event-details/fe435e44...
3,2648,2471,Azamat Murzakanov,Alonzo Menifield,Light Heavyweight Bout,35.0,36.0,177.80,182.88,1,...,0,2,0,2,0,498,498,2648,http://www.ufcstats.com/fight-details/fbbb7035...,http://www.ufcstats.com/event-details/fe435e44...
4,997,3856,Sedriques Dumas,Denis Tiuliulin,Middleweight Bout,28.0,36.0,187.96,185.42,0,...,2,8,1,8,0,900,900,997,http://www.ufcstats.com/fight-details/e6851877...,http://www.ufcstats.com/event-details/fe435e44...


## 3. Handle Missing Values

In [25]:
# fighters.csv also has NaN for these fighters — can't use to fill in missing values.
# Strategy:
#   height    -> drop rows (only 16 rows, height is meaningful so don't impute)
#   age       -> fill with median (less critical, small number missing)
#   *** CONNOR DOES THIS MAKE SENSE??? ****: ctrl_time -> fill with 0 (no control recorded = 0 seconds) 

# --- Height: drop rows with missing height ---
before = len(fights)
fights.dropna(subset=['f1_height_cm', 'f2_height_cm'], inplace=True)
print(f'Dropped {before - len(fights)} rows with missing height. Remaining: {len(fights)}')

# --- Age: fill with median ---
fights['f1_age_during'] = fights['f1_age_during'].fillna(fights['f1_age_during'].median())
fights['f2_age_during'] = fights['f2_age_during'].fillna(fights['f2_age_during'].median())

# --- Control time: fill with 0 ---
fights['f1_ctrl_time'] = fights['f1_ctrl_time'].fillna(0)
fights['f2_ctrl_time'] = fights['f2_ctrl_time'].fillna(0)

# Verify — all should be 0
print(fights[['f1_height_cm','f2_height_cm','f1_age_during','f2_age_during','f1_ctrl_time','f2_ctrl_time']].isnull().sum())

Dropped 0 rows with missing height. Remaining: 7824
f1_height_cm     0
f2_height_cm     0
f1_age_during    0
f2_age_during    0
f1_ctrl_time     0
f2_ctrl_time     0
dtype: int64


## 4. Fix Data Types
Make sure columns are the right type (e.g., dates as datetime, IDs as int).

In [ ]:
# TODO: convert types as needed
# Example: events['event_date'] = pd.to_datetime(events['event_date'])


## 5. Clean the Target Column (`result`)
`result` currently holds the winner's fighter_id. Convert it to a binary label: 1 if f1 won, 0 if f2 won.

In [ ]:
# TODO: create a binary target column
# Example: fights['f1_win'] = (fights['result'] == fights['f1_id']).astype(int)
# Check value counts after


## 6. Remove Irrelevant Columns
Drop URL columns and any other columns that won't be used as features.

In [ ]:
# TODO: drop columns you don't need
# Example: fights.drop(columns=['fights_url', 'event_url', 'result'], inplace=True)


## 7. Handle Categorical Columns
`weight_class` is a string — encode it for modeling.

In [ ]:
# Check unique weight classes
fights['weight_class'].value_counts()

In [ ]:
# TODO: encode weight_class (label encode or one-hot)
# Example: fights = pd.get_dummies(fights, columns=['weight_class'])


## 8. Check for Duplicates

In [ ]:
# TODO: find and drop duplicate rows
print(fights.duplicated().sum())
# fights.drop_duplicates(inplace=True)


## 9. Check for Outliers
Look at numeric columns for extreme values that may be errors.

In [ ]:
# Basic stats to spot outliers
fights.describe()

In [ ]:
# TODO: investigate and handle suspicious values


## 10. Save Cleaned Data

In [ ]:
# TODO: save the cleaned dataframe
# fights.to_csv('../data/processed/fights_cleaned.csv', index=False)
print('Done. Shape:', fights.shape)